# Silver: parse the PDFs with `ai_parse_document`

Task `silver_parse` of the job `ingest_financebench`. Runs the AI parser on the parts in `bronze_pdf_parts` and saves
the result in `silver_parsed`, **one row per part**:

| Column | Holds |
| --- | --- |
| `doc_name` | The document name, from the file name: `3M_2018_10K` for `.../pdfs/3M_2018_10K.pdf`. It is the `doc_name` FinanceBench's questions use |
| `path` | The PDF in the volume |
| `part_no`, `page_offset` | Which part of the PDF (bronze cuts long PDFs into parts of 50 pages), and how many pages come before it |
| `parsed` | The parser's output (VARIANT): the `elements` in reading order, each with a `type` (title, section_header, text, table as HTML, ...), its `content` and its page in `bbox[0].page_id`. The real page in the PDF is `page_offset + page_id + 1`. `error_status` lists the pages the parser could not convert |
| `parsed_at` | When it was parsed |

**Repairs itself on a rerun.** The parser does not fail when it cannot convert a page: it returns the part with that
page missing and the reason in `error_status`. When the reason is its rate limit (`RETRY_ERROR`), the page would
convert on a second try. So every run parses:

- the parts that are not in `silver_parsed` yet, and
- the parts whose stored parse lost a page to the rate limit,

and writes them with a `MERGE` on `path` and `part_no`: a new part is inserted, a repaired part replaces its old row.
A part parsed without a rate-limit error is never parsed again. A file the parser cannot convert at all
("Document conversion failed.") is stored with its `error_status` and not tried again, since a retry would fail the
same way and be billed every run. A rerun with nothing new and nothing to repair parses nothing.

**Change data feed is on**, so `gold_chunk` sees both the inserted and the replaced rows and chunks those documents
again.

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [ ]:
CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.finance_bench_bronze"
SILVER_SCHEMA = f"{CATALOG}.finance_bench_silver"

PARTS_TABLE = f"{BRONZE_SCHEMA}.bronze_pdf_parts"
PARSED_TABLE = f"{SILVER_SCHEMA}.silver_parsed"
STAGING_TABLE = f"{SILVER_SCHEMA}._parse_staging"  # this run's parses, so the MERGE never calls the parser twice

RETRY_ERROR = "Request rate limit exceeded"  # a page lost to this is parsed again by the next run

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARSED_TABLE} (
  doc_name STRING, path STRING, part_no INT, page_offset INT, parsed VARIANT, parsed_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
spark.sql(f"ALTER TABLE {PARSED_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")  # a table made before

## Step 1: the parts to parse

Every part of `bronze_pdf_parts` that has no good parse in `silver_parsed`: a `left_anti` join against the stored
parses that lost no page to the rate limit. Those are the new parts and the parts to repair.

**Example.** `bronze_pdf_parts` has 4 parts. In `silver_parsed`, `PGE_2020_10K` part 1 parsed cleanly, `NIKE_2018_10K`
part 1 lost 74 pages to the rate limit, `INTEL_2023_8K_dated-2023-02-10` could not be converted at all, and
`AMD_2022_10K` part 1 is new since the last run.

**1a. `silver_parsed` before**, with what its `error_status` says:

| doc_name | part_no | parsed | error_status |
| --- | --- | --- | --- |
| PGE_2020_10K | 1 | `{"document": {"elements": [{"type": "title", …}, …]}, …}` | none |
| NIKE_2018_10K | 1 | `{"document": {"elements": [… 152 elements …]}, "error_status": [{"error_message": "Request rate limit exceeded. Please try again later.", "page_id": 0}, …], …}` | 74 pages, `Request rate limit exceeded…` |
| INTEL_2023_8K_dated-2023-02-10 | 1 | `{"document": {"elements": []}, "error_status": [{"error_message": "Document conversion failed.", "page_id": 0}], …}` | `Document conversion failed.` |

**1b. After the `filter`.** The stored parses to keep: every one without `RETRY_ERROR`. `NIKE_2018_10K` is left out, so
it is parsed again. `INTEL…` stays, so it is not:

| path | part_no |
| --- | --- |
| …/pdfs/PGE_2020_10K.pdf | 1 |
| …/pdfs/INTEL_2023_8K_dated-2023-02-10.pdf | 1 |

**1c. After the `left_anti` join.** The parts of `bronze_pdf_parts` that are not in 1b:

| path | part_no | page_offset | pages | content |
| --- | --- | --- | --- | --- |
| …/pdfs/NIKE_2018_10K.pdf | 1 | 0 | 97 | `25 50 44 46 …` |
| …/pdfs/AMD_2022_10K.pdf | 1 | 0 | 42 | `25 50 44 46 …` |

In [ ]:
retry = F.expr(f"""exists(cast(parsed:error_status AS array<variant>),
                          e -> variant_get(e, '$.error_message', 'string') LIKE '{RETRY_ERROR}%')""")
todo = spark.table(PARTS_TABLE).join(
    spark.table(PARSED_TABLE).filter(~retry).select("path", "part_no"), ["path", "part_no"], "left_anti")

## Step 2: parse them

Each part of Step 1 goes through `ai_parse_document` once, and the results are written to the staging table, which
is replaced on every run. Writing them first means the parser runs exactly once per part: a `MERGE` can read its
source more than once, and each read of the parser would be billed again.

**Example.** Continues from Step 1.

**2a. After `select`, written to `_parse_staging`.** The document name comes from the file name, and the bytes become
the parser's output. This time `NIKE_2018_10K` parsed with no page lost:

| doc_name | path | part_no | page_offset | parsed | parsed_at |
| --- | --- | --- | --- | --- | --- |
| NIKE_2018_10K | …/pdfs/NIKE_2018_10K.pdf | 1 | 0 | `{"document": {"elements": [… 611 elements …]}, …}` | 2026-11-27 10:02:41 |
| AMD_2022_10K | …/pdfs/AMD_2022_10K.pdf | 1 | 0 | `{"document": {"elements": [{"type": "title", "content": "UNITED STATES⏎SECURITIES…", "bbox": [{"page_id": 0, …}]}, …]}, …}` | 2026-11-27 10:02:41 |

In [ ]:
(
    todo.select(
        F.regexp_extract("path", r"([^/]+)\.pdf$", 1).alias("doc_name"),  # .../pdfs/3M_2018_10K.pdf -> 3M_2018_10K
        "path", "part_no", "page_offset",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
        F.current_timestamp().alias("parsed_at"),
    )
    .write.mode("overwrite").saveAsTable(STAGING_TABLE)
)

## Step 3: merge them into `silver_parsed`

A `MERGE` on `path` and `part_no`: a part that is already there (one being repaired) is updated with its new parse, a
new part is inserted. The other rows are not touched. With change data feed, an update reaches `gold_chunk` as an
`update_postimage` row and an insert as an `insert` row.

**Example.** Continues from Step 2.

**3a. After `MERGE`, `silver_parsed`:**

| doc_name | part_no | parsed | parsed_at | |
| --- | --- | --- | --- | --- |
| PGE_2020_10K | 1 | `{"document": {"elements": [{"type": "title", …}, …]}, …}` | 2026-11-20 09:31:47 | not touched |
| NIKE_2018_10K | 1 | `{"document": {"elements": [… 611 elements …]}, …}` | 2026-11-27 10:02:41 | **updated**: repaired |
| INTEL_2023_8K_dated-2023-02-10 | 1 | `{"document": {"elements": []}, "error_status": [{"error_message": "Document conversion failed.", …}], …}` | 2026-11-24 16:33:00 | not touched: not retried |
| AMD_2022_10K | 1 | `{"document": {"elements": […]}, …}` | 2026-11-27 10:02:41 | **inserted**: new |

If a page of `NIKE_2018_10K` had hit the rate limit again, its row would still be replaced, with fewer pages lost,
and the next run would try it again. A rerun now finds nothing in Step 1 and parses nothing.

In [ ]:
(
    DeltaTable.forName(spark, PARSED_TABLE).alias("t")
    .merge(spark.table(STAGING_TABLE).alias("s"), "t.path = s.path AND t.part_no = s.part_no")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
display(spark.sql(f"""
SELECT count(*) AS parts, count(DISTINCT doc_name) AS documents,
       count_if(exists(cast(parsed:error_status AS array<variant>),
                       e -> variant_get(e, '$.error_message', 'string') LIKE '{RETRY_ERROR}%')) AS parts_to_repair,
       count_if(size(cast(parsed:document:elements AS array<variant>)) = 0) AS parts_not_converted
FROM {PARSED_TABLE}"""))